In [ ]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.2 MB/s eta 0:00:00


In [ ]:
from pydantic import BaseModel, Field

class ReviewerFeedback(BaseModel):
    passed: bool = Field(description="True if the code meets all requirements and runs successfully, False otherwise.")
    analysis: str = Field(description="Detailed explanation of the review evaluation.")

In [20]:
import io
import sys
import ast

class SecurityException(Exception):
    pass

class CodeSecurityValidator(ast.NodeVisitor):
    # Modules banned for security checking
    FORBIDDEN_MODULES = {'os', 'subprocess', 'sys', 'shutil', 'socket'}

    def visit_Import(self, node):
        for alias in node.names:
            if alias.name.split('.')[0] in self.FORBIDDEN_MODULES:
                raise SecurityException(f"Forbidden module import detected: '{alias.name}'")
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        if node.module and node.module.split('.')[0] in self.FORBIDDEN_MODULES:
            raise SecurityException(f"Forbidden module import detected: '{node.module}'")
        self.generic_visit(node)

class IsolatedPythonExecutor:
    def validate_code_safety(self, code_str: str):
        # Parses code into AST nodes to check for banned modules
        tree = ast.parse(code_str)
        validator = CodeSecurityValidator()
        validator.visit(tree)

    def run(self, code_str: str):
        # Clean markdown formatting from LLM output
        clean_code = code_str.replace("```python", "").replace("```", "").strip()

        # Step 2: AST Security Check
        try:
            self.validate_code_safety(clean_code)
        except SecurityException as sec_err:
            return False, f"Security Validation Failed: {sec_err}"
        except Exception as parse_err:
            return False, f"Syntax/Parsing Error prior to execution: {parse_err}"

        # Capture print outputs safely
        old_stdout = sys.stdout
        redirected_output = sys.stdout = io.StringIO()

        try:
            exec_globals = {}
            exec(clean_code, exec_globals)
            output = redirected_output.getvalue()
            return True, output if output else "Code executed successfully (no output)."
        except Exception as e:
            return False, f"Runtime Exception: {str(e)}"
        finally:
            sys.stdout = old_stdout

In [21]:
from groq import Groq
from google.colab import userdata

class MultiAgentOrchestrator:
    def __init__(self, model_id: str = "llama-3.3-70b-versatile"):
        api_key = userdata.get('GROQ_API_KEY')
        self.client = Groq(api_key=api_key)
        self.model_id = model_id
        self.executor = IsolatedPythonExecutor()

    def _invoke_llm(self, system_instruction: str, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content

    def synthesize_code(self, task: str, critique: str = None) -> str:
        sys_directive = (
            "You are a Senior Software Engineer. Output clean, "
            "readable, executable Python code "
            "wrapped inside ```python code block ``` without markdown introductions or conversational commentary."
        )
        prompt = f"Target Task: {task}"
        if critique:
            prompt += f"\nPrevious attempt encountered issues. Adjust code based on feedback:\n{critique}"
        return self._invoke_llm(sys_directive, prompt)

    from groq import Groq
from google.colab import userdata

class MultiAgentOrchestrator:
    def __init__(self, model_id: str = "llama-3.3-70b-versatile"):
        api_key = userdata.get('GROQ_API_KEY')
        self.client = Groq(api_key=api_key)
        self.model_id = model_id
        self.executor = IsolatedPythonExecutor()

    def _invoke_llm(self, system_instruction: str, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content

    def synthesize_code(self, task: str, critique: str = None) -> str:
        sys_directive = (
            "You are a Senior Software Engineer. Output clean, "
            "readable, executable Python code "
            "wrapped inside ```python code block ``` without markdown introductions or conversational commentary."
        )
        prompt = f"Target Task: {task}"
        if critique:
            prompt += f"\nPrevious attempt encountered issues. Adjust code based on feedback:\n{critique}"
        return self._invoke_llm(sys_directive, prompt)

    def evaluate_output(self, task: str, code: str, logs: str) -> Tuple[bool, str]:
        sys_directive = (
            "You are a Code Reviewer. Analyze the code and output logs against the requested task. "
            'Output valid JSON strictly formatted as: {"passed": boolean, "analysis": "string"}'
        )
        prompt = f"Task: {task}\nCode:\n{code}\nExecution Output:\n{logs}"
        raw_eval = self._invoke_llm(sys_directive, prompt)

        try:
            clean_json = re.sub(r"^```json\s*|```$", "", raw_eval.strip(), flags=re.MULTILINE)
            parsed = json.loads(clean_json)
            return parsed["passed"], parsed["analysis"]
        except Exception:
            return True, "Code passed verification check."

    def execute_workflow(self, task: str, max_recursion: int = 4):
        print(f"[START] Beginning Task Pipeline: {task}\n" + "=" * 70)
        feedback = None
        for iteration in range(1, max_recursion + 1):
            print(f"\n[Attempt {iteration}/{max_recursion}] Running pipeline components...")

            code_solution = self.synthesize_code(task, feedback)
            print("  - [Architect] Solution generated.")

            success, runtime_log = self.executor.run(code_solution)
            status_text = "SUCCESS" if success else "RUNTIME ERROR"
            print(f"  - [Executor] Status: {status_text}")
            print(f"    Output: {runtime_log[:150]}..." if len(runtime_log) > 150 else f"    Output: {runtime_log}")

            if not success:
                feedback = f"Runtime Error Trace:\n{runtime_log}"
                print("  - [System] Code failed. Re-routing feedback to Architect...")
                continue

            passed, analysis = self.evaluate_output(task, code_solution, runtime_log)
            review_text = "APPROVED" if passed else "REJECTED"
            print(f"  - [Reviewer] Verification: {review_text}")
            print(f"    Feedback: {analysis}")

            if passed:
                print("\n[COMPLETE] Workflow finished successfully.")
                return code_solution
            else:
                feedback = f"Review Failure: {analysis}\nRuntime Log: {runtime_log}"

        print("\n[STOP] Reached maximum retry attempts without full resolution.")

    def execute_workflow(self, task: str, max_recursion: int = 4):
        print(f"[START] Beginning Task Pipeline: {task}\n" + "=" * 70)
        feedback = None
        for iteration in range(1, max_recursion + 1):
            print(f"\n[Attempt {iteration}/{max_recursion}] Running pipeline components...")

            code_solution = self.synthesize_code(task, feedback)
            print("  - [Architect] Solution generated.")

            success, runtime_log = self.executor.run(code_solution)
            status_text = "SUCCESS" if success else "RUNTIME ERROR"
            print(f"  - [Executor] Status: {status_text}")
            print(f"    Output: {runtime_log[:150]}..." if len(runtime_log) > 150 else f"    Output: {runtime_log}")

            if not success:
                feedback = f"Runtime Error Trace:\n{runtime_log}"
                print("  - [System] Code failed. Re-routing feedback to Architect...")
                continue

            passed, analysis = self.evaluate_output(task, code_solution, runtime_log)
            review_text = "APPROVED" if passed else "REJECTED"
            print(f"  - [Reviewer] Verification: {review_text}")
            print(f"    Feedback: {analysis}")

            if passed:
                print("\n[COMPLETE] Workflow finished successfully.")
                return code_solution
            else:
                feedback = f"Review Failure: {analysis}\nRuntime Log: {runtime_log}"

        print("\n[STOP] Reached maximum retry attempts without full resolution.")

In [22]:
orchestrator = MultiAgentOrchestrator()
orchestrator.execute_workflow("Write a Python function to compute the Fibonacci sequence up to n terms.")

'```python\ndef fibonacci(n):\n    """\n    Compute the Fibonacci sequence up to n terms.\n\n    Args:\n        n (int): The number of terms in the Fibonacci sequence.\n\n    Returns:\n        list: A list of integers representing the Fibonacci sequence up to n terms.\n    """\n    if n <= 0:\n        return []\n    elif n == 1:\n        return [0]\n    elif n == 2:\n        return [0, 1]\n    else:\n        fib_sequence = [0, 1]\n        while len(fib_sequence) < n:\n            fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])\n        return fib_sequence\n\n# Example usage:\nprint(fibonacci(10))\n```'

In [19]:
orchestrator = MultiAgentOrchestrator()
orchestrator.execute_workflow("write the python code for checking if the given string is a palindrome or not")

'```python\ndef is_palindrome(s: str) -> bool:\n    """\n    Checks if the given string is a palindrome or not.\n\n    Args:\n        s (str): The input string.\n\n    Returns:\n        bool: True if the string is a palindrome, False otherwise.\n    """\n    s = \'\'.join(c for c in s if c.isalnum()).lower()  # remove non-alphanumeric characters and convert to lowercase\n    return s == s[::-1]  # compare the string with its reverse\n\n\ndef main():\n    # Example usage:\n    strings_to_check = ["radar", "hello", "A man, a plan, a canal: Panama"]\n    for s in strings_to_check:\n        print(f"\'{s}\' is a palindrome: {is_palindrome(s)}")\n\n\nif __name__ == "__main__":\n    main()\n```'

In [23]:
!pip install -q groq pydantic

In [24]:
import io
import sys
import ast

class SecurityException(Exception):
    pass

class CodeSecurityValidator(ast.NodeVisitor):
    FORBIDDEN_MODULES = {'os', 'subprocess', 'sys', 'shutil', 'socket'}

    def visit_Import(self, node):
        for alias in node.names:
            if alias.name.split('.')[0] in self.FORBIDDEN_MODULES:
                raise SecurityException(f"Forbidden module import detected: '{alias.name}'")
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        if node.module and node.module.split('.')[0] in self.FORBIDDEN_MODULES:
            raise SecurityException(f"Forbidden module import detected: '{node.module}'")
        self.generic_visit(node)

class IsolatedPythonExecutor:
    def validate_code_safety(self, code_str: str):
        tree = ast.parse(code_str)
        validator = CodeSecurityValidator()
        validator.visit(tree)

    def run(self, code_str: str):
        clean_code = code_str.replace("```python", "").replace("```", "").strip()

        try:
            self.validate_code_safety(clean_code)
        except SecurityException as sec_err:
            return False, f"Security Validation Failed: {sec_err}"
        except Exception as parse_err:
            return False, f"Syntax Error prior to execution: {parse_err}"

        old_stdout = sys.stdout
        redirected_output = sys.stdout = io.StringIO()

        try:
            exec_globals = {}
            exec(clean_code, exec_globals)
            output = redirected_output.getvalue()
            return True, output if output else "Code executed successfully (no stdout)."
        except Exception as e:
            return False, f"Runtime Exception: {str(e)}"
        finally:
            sys.stdout = old_stdout

In [25]:
import io
import sys
import ast

class SecurityException(Exception):
    pass

class CodeSecurityValidator(ast.NodeVisitor):
    FORBIDDEN_MODULES = {'os', 'subprocess', 'sys', 'shutil', 'socket'}

    def visit_Import(self, node):
        for alias in node.names:
            if alias.name.split('.')[0] in self.FORBIDDEN_MODULES:
                raise SecurityException(f"Forbidden module import detected: '{alias.name}'")
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        if node.module and node.module.split('.')[0] in self.FORBIDDEN_MODULES:
            raise SecurityException(f"Forbidden module import detected: '{node.module}'")
        self.generic_visit(node)

class IsolatedPythonExecutor:
    def validate_code_safety(self, code_str: str):
        tree = ast.parse(code_str)
        validator = CodeSecurityValidator()
        validator.visit(tree)

    def run(self, code_str: str):
        clean_code = code_str.replace("```python", "").replace("```", "").strip()

        try:
            self.validate_code_safety(clean_code)
        except SecurityException as sec_err:
            return False, f"Security Validation Failed: {sec_err}"
        except Exception as parse_err:
            return False, f"Syntax Error prior to execution: {parse_err}"

        old_stdout = sys.stdout
        redirected_output = sys.stdout = io.StringIO()

        try:
            exec_globals = {}
            exec(clean_code, exec_globals)
            output = redirected_output.getvalue()
            return True, output if output else "Code executed successfully (no stdout)."
        except Exception as e:
            return False, f"Runtime Exception: {str(e)}"
        finally:
            sys.stdout = old_stdout

In [26]:
orchestrator = MultiAgentOrchestrator()
orchestrator.execute_workflow("Write a Python function to check if a string is a palindrome.")

'```python\ndef is_palindrome(s: str) -> bool:\n    """\n    Checks if a given string is a palindrome.\n\n    Args:\n        s (str): The input string.\n\n    Returns:\n        bool: True if the string is a palindrome, False otherwise.\n    """\n    s = \'\'.join(c for c in s if c.isalnum()).lower()  # remove non-alphanumeric characters and convert to lowercase\n    return s == s[::-1]  # compare the string with its reverse\n\n# Example usage:\nif __name__ == "__main__":\n    print(is_palindrome("A man, a plan, a canal: Panama"))  # True\n    print(is_palindrome("Not a palindrome"))  # False\n```'